# Experiment Comparison: Pretrained vs Random × Single-task vs Multi-task

This notebook compares predictions from the four Golem fine-tuning experiments:

| Experiment | Training | Pretrained? | Endpoints |
|------------|----------|-------------|----------|
| `st_random` | Single-task | No | LogD only |
| `st_pretrained` | Single-task | Yes (MDAE) | LogD only |
| `mt_random` | Multi-task | No | All 9 |
| `mt_pretrained` | Multi-task | Yes (MDAE) | All 9 |

## Research Questions

1. Does MDAE pretraining improve fine-tuning performance?
2. Does multi-task learning outperform single-task on shared endpoints?
3. Do pretraining benefits vary by endpoint?

## 1. Configuration

In [ ]:
from pathlib import Path

# Ground truth test set
GROUND_TRUTH_PATH = Path('../data/test-set/expansion_data_test_full_lb_flag.csv')

# Experiment directories containing predictions_log_scale.csv
EXPERIMENTS_DIR = Path('../experiments')

EXPERIMENTS = {
    'ST Random':     {'dir': 'st_random',      'training': 'Single-task', 'pretrained': False},
    'ST Pretrained': {'dir': 'st_pretrained',   'training': 'Single-task', 'pretrained': True},
    'MT Random':     {'dir': 'mt_random',       'training': 'Multi-task',  'pretrained': False},
    'MT Pretrained': {'dir': 'mt_pretrained',   'training': 'Multi-task',  'pretrained': True},
}

# Bootstrap samples for confidence intervals
N_BOOTSTRAP = 1000

print(f'Ground truth: {GROUND_TRUTH_PATH}')
print(f'Experiments:  {list(EXPERIMENTS.keys())}')

## 2. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Optional, Tuple
from scipy.stats import spearmanr, kendalltau
from sklearn.metrics import mean_absolute_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.precision', 4)

print('Setup complete!')

## 3. Constants and Evaluation Functions

In [ ]:
# All 9 log-scale endpoints (as named in prediction CSVs)
ALL_LOG_ENDPOINTS = [
    'LogD', 'LogS', 'Log_HLM_CLint', 'Log_MLM_CLint',
    'Log_Caco_Papp_AB', 'Log_Caco_ER',
    'Log_Mouse_PPB', 'Log_Mouse_BPB', 'Log_Mouse_MPB',
]

# Short names for plots
ENDPOINT_SHORT = {
    'LogD': 'LogD', 'LogS': 'LogS',
    'Log_HLM_CLint': 'HLM', 'Log_MLM_CLint': 'MLM',
    'Log_Caco_Papp_AB': 'Caco2-Papp', 'Log_Caco_ER': 'Caco2-ER',
    'Log_Mouse_PPB': 'PPB', 'Log_Mouse_BPB': 'BPB', 'Log_Mouse_MPB': 'MPB',
}

# Map from ground-truth original column names to log-scale column names
# Used to rename GT columns so they match the prediction CSVs
ENDPOINT_MAP = {
    'LogD':                          ('LogD', 1),
    'KSOL':                          ('LogS', 1e-6),
    'HLM CLint':                     ('Log_HLM_CLint', 1),
    'MLM CLint':                     ('Log_MLM_CLint', 1),
    'Caco-2 Permeability Papp A>B':  ('Log_Caco_Papp_AB', 1e-6),
    'Caco-2 Permeability Efflux':    ('Log_Caco_ER', 1),
    'MPPB':                          ('Log_Mouse_PPB', 1),
    'MBPB':                          ('Log_Mouse_BPB', 1),
    'MGMB':                          ('Log_Mouse_MPB', 1),
}

METRICS = ['MAE', 'RAE', 'R2', 'Spearman', 'Kendall']
LOWER_IS_BETTER = ['MAE', 'RAE']

print(f'Endpoints: {len(ALL_LOG_ENDPOINTS)}')
print(f'Metrics: {METRICS}')

In [ ]:
# === Evaluation functions ===

def compute_metrics(pred: np.ndarray, true: np.ndarray) -> Dict[str, float]:
    """Calculate all evaluation metrics for aligned arrays."""
    mae = mean_absolute_error(true, pred)
    rae = mae / np.mean(np.abs(true - np.mean(true))) if np.std(true) > 0 else np.nan
    r2 = r2_score(true, pred) if np.std(true) > 0 else np.nan
    spr = spearmanr(true, pred).statistic if np.std(pred) > 1e-6 else np.nan
    ktau = kendalltau(true, pred).statistic if np.std(pred) > 1e-6 else np.nan
    return {'MAE': mae, 'RAE': rae, 'R2': r2, 'Spearman': spr, 'Kendall': ktau}


def bootstrap_evaluate(
    y_pred: np.ndarray, y_true: np.ndarray,
    n_bootstrap: int = 1000, seed: int = 42,
) -> pd.DataFrame:
    """Bootstrap evaluation: returns DataFrame with one row per sample."""
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n_bootstrap):
        idx = rng.choice(len(y_true), size=len(y_true), replace=True)
        m = compute_metrics(y_pred[idx], y_true[idx])
        m['sample'] = i
        rows.append(m)
    return pd.DataFrame(rows)


def bootstrap_significance(
    bs1: pd.DataFrame, bs2: pd.DataFrame, metric: str,
) -> Tuple[float, bool]:
    """Test if model2 (bs2) is significantly better than model1 (bs1)."""
    diff = bs2[metric].values - bs1[metric].values
    if metric in LOWER_IS_BETTER:
        p_value = np.mean(diff >= 0)   # fraction where model2 is NOT better
        is_better = np.mean(diff) < 0
    else:
        p_value = np.mean(diff <= 0)
        is_better = np.mean(diff) > 0
    return p_value, is_better


print('Evaluation functions loaded!')

## 4. Load Data

In [ ]:
# === Load ground truth ===

ground_truth = pd.read_csv(GROUND_TRUTH_PATH)
print(f'Ground truth: {len(ground_truth)} molecules')

# Rename ground-truth columns to match prediction log-scale column names
GT_COL_RENAME = {orig: log for orig, (log, _) in ENDPOINT_MAP.items() if orig != log}
ground_truth = ground_truth.rename(columns=GT_COL_RENAME)

print(f'\nSamples per endpoint:')
for ep in ALL_LOG_ENDPOINTS:
    if ep in ground_truth.columns:
        count = ground_truth[ep].notna().sum()
        print(f'  {ENDPOINT_SHORT[ep]:12s} ({ep}): {count}')
    else:
        print(f'  {ENDPOINT_SHORT[ep]:12s} ({ep}): NOT FOUND')

In [ ]:
# === Load predictions from each experiment ===

predictions = {}
for name, cfg in EXPERIMENTS.items():
    path = EXPERIMENTS_DIR / cfg['dir'] / 'predictions_log_scale.csv'
    if not path.exists():
        print(f'WARNING: {path} not found — skipping {name}')
        continue
    df = pd.read_csv(path)
    predictions[name] = df
    ep_cols = [c for c in df.columns if c in ALL_LOG_ENDPOINTS]
    print(f'{name:16s}: {len(df)} molecules, {len(ep_cols)} endpoints {ep_cols}')

print(f'\nLoaded {len(predictions)} experiments')

## 5. Evaluate All Experiments

In [ ]:
# === Evaluate each experiment on each available endpoint ===

# Store: results[model_name][endpoint] = bootstrap DataFrame
bs_results: Dict[str, Dict[str, pd.DataFrame]] = {}
# Store point metrics: point_metrics[model_name][endpoint] = {metric: (mean, std)}
point_metrics: Dict[str, Dict[str, Dict[str, Tuple[float, float]]]] = {}
# Store aligned arrays for scatter plots
aligned_data: Dict[str, Dict[str, Tuple[np.ndarray, np.ndarray]]] = {}

for model_name, pred_df in predictions.items():
    bs_results[model_name] = {}
    point_metrics[model_name] = {}
    aligned_data[model_name] = {}

    pred_endpoints = [c for c in pred_df.columns if c in ALL_LOG_ENDPOINTS]

    for ep in pred_endpoints:
        # Merge on Molecule Name
        merged = pred_df[['Molecule Name', ep]].merge(
            ground_truth[['Molecule Name', ep]],
            on='Molecule Name', suffixes=('_pred', '_true'),
        )
        subset = merged[[f'{ep}_pred', f'{ep}_true']].dropna()
        if len(subset) < 10:
            continue

        y_pred = subset[f'{ep}_pred'].to_numpy()
        y_true = subset[f'{ep}_true'].to_numpy()

        aligned_data[model_name][ep] = (y_pred, y_true)

        bs = bootstrap_evaluate(y_pred, y_true, n_bootstrap=N_BOOTSTRAP)
        bs_results[model_name][ep] = bs
        point_metrics[model_name][ep] = {
            m: (bs[m].mean(), bs[m].std()) for m in METRICS
        }

    print(f'{model_name:16s}: evaluated on {len(bs_results[model_name])} endpoints')

print('\nDone!')

## 6. Results Summary Tables

In [ ]:
def build_summary_table(metric: str = 'MAE') -> pd.DataFrame:
    """Build a comparison table for one metric across all models and endpoints."""
    rows = []
    for ep in ALL_LOG_ENDPOINTS:
        row = {'Endpoint': ENDPOINT_SHORT[ep]}
        for model_name in predictions:
            if ep in point_metrics.get(model_name, {}):
                mean, std = point_metrics[model_name][ep][metric]
                row[model_name] = f'{mean:.3f} \u00b1 {std:.3f}'
            else:
                row[model_name] = '-'
        rows.append(row)

    # Macro average (over endpoints each model has)
    row = {'Endpoint': 'Macro Avg'}
    for model_name in predictions:
        vals = [
            point_metrics[model_name][ep][metric][0]
            for ep in ALL_LOG_ENDPOINTS
            if ep in point_metrics.get(model_name, {})
        ]
        if vals:
            row[model_name] = f'{np.mean(vals):.3f}'
        else:
            row[model_name] = '-'
    rows.append(row)

    return pd.DataFrame(rows).set_index('Endpoint')


# Display comparison tables
for metric in METRICS:
    print(f"\n{'='*70}")
    print(f'{metric} Comparison')
    print('='*70)
    display(build_summary_table(metric))

In [ ]:
# === Compact macro-average summary ===

macro_rows = []
for model_name in predictions:
    cfg = EXPERIMENTS[model_name]
    row = {
        'Model': model_name,
        'Training': cfg['training'],
        'Pretrained': cfg['pretrained'],
    }
    for metric in METRICS:
        vals = [
            point_metrics[model_name][ep][metric][0]
            for ep in ALL_LOG_ENDPOINTS
            if ep in point_metrics.get(model_name, {})
        ]
        row[metric] = f'{np.mean(vals):.4f}' if vals else '-'
    row['# Endpoints'] = len(point_metrics.get(model_name, {}))
    macro_rows.append(row)

macro_df = pd.DataFrame(macro_rows)
print('\nMacro-Averaged Summary')
print('='*80)
display(macro_df.set_index('Model'))

## 7. Pretraining Improvement Analysis

In [ ]:
def calculate_improvement(
    baseline_name: str, improved_name: str,
) -> pd.DataFrame:
    """
    Calculate per-endpoint improvement from baseline to improved model.

    For MAE/RAE: negative delta = improvement (lower is better).
    For R2/correlations: positive delta = improvement (higher is better).
    """
    rows = []
    # Only compare on shared endpoints
    shared_eps = sorted(
        set(point_metrics.get(baseline_name, {}).keys())
        & set(point_metrics.get(improved_name, {}).keys())
    )

    for ep in shared_eps:
        row = {'Endpoint': ENDPOINT_SHORT[ep], 'Endpoint_Full': ep}
        for metric in METRICS:
            base_val = point_metrics[baseline_name][ep][metric][0]
            impr_val = point_metrics[improved_name][ep][metric][0]
            delta = impr_val - base_val
            pct = (delta / abs(base_val)) * 100 if base_val != 0 else np.nan
            is_better = delta < 0 if metric in LOWER_IS_BETTER else delta > 0

            row[f'{metric}_base'] = base_val
            row[f'{metric}_impr'] = impr_val
            row[f'{metric}_delta'] = delta
            row[f'{metric}_pct'] = pct
            row[f'{metric}_better'] = is_better
        rows.append(row)

    return pd.DataFrame(rows)


# --- Pretraining effect ---
comparisons = [
    ('ST Random', 'ST Pretrained', 'Single-task: Pretrained vs Random'),
    ('MT Random', 'MT Pretrained', 'Multi-task: Pretrained vs Random'),
]

improvement_dfs = {}
for baseline, improved, title in comparisons:
    if baseline not in predictions or improved not in predictions:
        continue
    df = calculate_improvement(baseline, improved)
    improvement_dfs[title] = df

    print(f"\n{'='*70}")
    print(title)
    print('='*70)

    cols = ['Endpoint']
    for m in ['MAE', 'R2', 'Spearman']:
        cols.extend([f'{m}_base', f'{m}_impr', f'{m}_delta', f'{m}_better'])
    display(df[cols].set_index('Endpoint'))

In [ ]:
# === Print improvement summary ===

for title, df in improvement_dfs.items():
    print(f"\n{'='*70}")
    print(title)
    print('='*70)

    improved_eps = df[df['MAE_better'] == True]
    hurt_eps = df[df['MAE_better'] == False]

    print('\nEndpoints IMPROVED by pretraining (lower MAE):')
    for _, row in improved_eps.iterrows():
        print(f"  {row['Endpoint']:12s} | MAE: {row['MAE_base']:.3f} -> {row['MAE_impr']:.3f} ({row['MAE_pct']:+.1f}%)")

    if len(hurt_eps) > 0:
        print('\nEndpoints HURT by pretraining (higher MAE):')
        for _, row in hurt_eps.iterrows():
            print(f"  {row['Endpoint']:12s} | MAE: {row['MAE_base']:.3f} -> {row['MAE_impr']:.3f} ({row['MAE_pct']:+.1f}%)")

    n_better = len(improved_eps)
    n_total = len(df)
    print(f'\nSummary: {n_better}/{n_total} endpoints improved')

## 8. Visualizations

In [ ]:
def plot_metric_bars(metric: str = 'MAE', figsize: Tuple = (12, 6)):
    """Bar plot comparing a metric across endpoints for all models."""
    # Only show multi-task models (have all endpoints)
    mt_models = [n for n in predictions if EXPERIMENTS[n]['training'] == 'Multi-task']

    data = []
    for ep in ALL_LOG_ENDPOINTS:
        for model_name in mt_models:
            if ep in point_metrics.get(model_name, {}):
                mean, std = point_metrics[model_name][ep][metric]
                data.append({
                    'Endpoint': ENDPOINT_SHORT[ep],
                    'Model': model_name,
                    'Value': mean,
                    'Std': std,
                })

    plot_df = pd.DataFrame(data)
    if plot_df.empty:
        print(f'No data for {metric}')
        return

    fig, ax = plt.subplots(figsize=figsize)
    pivot = plot_df.pivot(index='Endpoint', columns='Model', values='Value')
    # Reorder endpoints
    ep_order = [ENDPOINT_SHORT[ep] for ep in ALL_LOG_ENDPOINTS]
    pivot = pivot.reindex([e for e in ep_order if e in pivot.index])
    pivot.plot(kind='bar', ax=ax, width=0.7, edgecolor='black', alpha=0.85)

    ax.set_xlabel('Endpoint', fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f'{metric} by Endpoint (Multi-task Models)', fontsize=14)
    ax.legend(title='Model', bbox_to_anchor=(1.02, 1), loc='upper left')
    ax.tick_params(axis='x', rotation=45)

    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=7, rotation=90, padding=3)

    plt.tight_layout()
    plt.show()


for metric in ['MAE', 'R2', 'Spearman']:
    plot_metric_bars(metric)

In [ ]:
# === LogD comparison across all 4 models ===

logd_models = [n for n in predictions if 'LogD' in point_metrics.get(n, {})]

if logd_models:
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    for ax, metric in zip(axes, ['MAE', 'R2', 'Spearman']):
        means = [point_metrics[n]['LogD'][metric][0] for n in logd_models]
        stds = [point_metrics[n]['LogD'][metric][1] for n in logd_models]
        colors = ['#3498db' if not EXPERIMENTS[n]['pretrained'] else '#e74c3c' for n in logd_models]

        bars = ax.bar(logd_models, means, yerr=stds, capsize=5,
                      color=colors, edgecolor='black', alpha=0.85)
        ax.set_ylabel(metric)
        ax.set_title(f'LogD: {metric}')
        ax.tick_params(axis='x', rotation=30)

        for bar, m in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f'{m:.3f}', ha='center', va='bottom', fontsize=9)

    from matplotlib.patches import Patch
    axes[0].legend(handles=[
        Patch(color='#3498db', label='Random'),
        Patch(color='#e74c3c', label='Pretrained'),
    ])

    plt.suptitle('LogD: All 4 Models', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# === Improvement heatmap (multi-task: pretrained vs random) ===

mt_improvement = improvement_dfs.get('Multi-task: Pretrained vs Random')

if mt_improvement is not None and len(mt_improvement) > 1:
    matrix_data = []
    for _, row in mt_improvement.iterrows():
        row_data = {'Endpoint': row['Endpoint']}
        for metric in METRICS:
            # Negate MAE/RAE so positive = improvement
            if metric in LOWER_IS_BETTER:
                row_data[metric] = -row[f'{metric}_pct']
            else:
                row_data[metric] = row[f'{metric}_pct']
        matrix_data.append(row_data)

    matrix_df = pd.DataFrame(matrix_data).set_index('Endpoint')

    fig, ax = plt.subplots(figsize=(10, 7))
    vmax = max(abs(matrix_df.values.min()), abs(matrix_df.values.max()))

    sns.heatmap(
        matrix_df, annot=True, fmt='.1f', cmap='RdYlGn',
        center=0, vmin=-vmax, vmax=vmax, ax=ax, linewidths=0.5,
        cbar_kws={'label': '% Improvement (positive = pretrained better)'},
    )

    ax.set_title('MT Pretrained vs MT Random — % Change per Metric\n(Green = Pretraining Helped)', fontsize=13)
    ax.set_xlabel('Metric', fontsize=12)
    ax.set_ylabel('Endpoint', fontsize=12)
    plt.tight_layout()
    plt.show()

## 9. Scatter Plots: Predicted vs True

In [ ]:
def plot_scatter_grid(endpoint: str, model_names: Optional[List[str]] = None,
                      figsize_per: Tuple = (4, 4)):
    """Scatter plots for one endpoint across selected models."""
    if model_names is None:
        model_names = [n for n in predictions if endpoint in aligned_data.get(n, {})]
    else:
        model_names = [n for n in model_names if endpoint in aligned_data.get(n, {})]

    if not model_names:
        print(f'No data for {endpoint}')
        return

    ncols = len(model_names)
    fig, axes = plt.subplots(1, ncols, figsize=(figsize_per[0] * ncols, figsize_per[1]))
    if ncols == 1:
        axes = [axes]

    for ax, name in zip(axes, model_names):
        y_pred, y_true = aligned_data[name][endpoint]
        ax.scatter(y_true, y_pred, alpha=0.4, s=15)
        lims = [
            min(y_true.min(), y_pred.min()),
            max(y_true.max(), y_pred.max()),
        ]
        ax.plot(lims, lims, 'r--', lw=2, alpha=0.7)

        mae = point_metrics[name][endpoint]['MAE'][0]
        r2 = point_metrics[name][endpoint]['R2'][0]

        ax.set_xlabel('True', fontsize=10)
        ax.set_ylabel('Predicted', fontsize=10)
        ax.set_title(f'{name}\nMAE={mae:.3f}, R\u00b2={r2:.3f}', fontsize=10)
        ax.set_aspect('equal', adjustable='box')

    fig.suptitle(f'{ENDPOINT_SHORT.get(endpoint, endpoint)}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


# Scatter plots for LogD (all 4 models)
plot_scatter_grid('LogD')

# Scatter plots for a few multi-task endpoints
for ep in ['LogS', 'Log_Caco_Papp_AB', 'Log_Mouse_PPB']:
    plot_scatter_grid(ep)

## 10. Statistical Significance (Bootstrap)

In [ ]:
# === Significance tests: pretrained vs random ===

sig_pairs = [
    ('ST Random', 'ST Pretrained'),
    ('MT Random', 'MT Pretrained'),
]

sig_rows = []
for base_name, impr_name in sig_pairs:
    if base_name not in bs_results or impr_name not in bs_results:
        continue
    shared_eps = sorted(
        set(bs_results[base_name].keys()) & set(bs_results[impr_name].keys())
    )
    for ep in shared_eps:
        for metric in ['MAE', 'R2', 'Spearman']:
            p_val, is_better = bootstrap_significance(
                bs_results[base_name][ep], bs_results[impr_name][ep], metric,
            )
            sig_rows.append({
                'Comparison': f'{impr_name} vs {base_name}',
                'Endpoint': ENDPOINT_SHORT[ep],
                'Metric': metric,
                'p_value': p_val,
                'Significant (p<0.05)': p_val < 0.05,
                'Pretrained Better': is_better,
            })

sig_df = pd.DataFrame(sig_rows)

if not sig_df.empty:
    print('Statistical Significance: Pretrained vs Random')
    print('='*80)
    display(
        sig_df.pivot_table(
            index=['Comparison', 'Endpoint'],
            columns='Metric',
            values=['p_value', 'Pretrained Better'],
        )
    )

In [ ]:
# === Summary: significant improvements ===

if not sig_df.empty:
    for comparison in sig_df['Comparison'].unique():
        print(f"\n{'='*70}")
        print(f'SIGNIFICANTLY BETTER (p < 0.05): {comparison}')
        print('='*70)

        subset = sig_df[
            (sig_df['Comparison'] == comparison)
            & (sig_df['Significant (p<0.05)'] == True)
            & (sig_df['Pretrained Better'] == True)
        ]

        if len(subset) == 0:
            print('  (none)')
        else:
            for metric in ['MAE', 'R2', 'Spearman']:
                eps = subset[subset['Metric'] == metric]
                if len(eps) > 0:
                    ep_list = ', '.join(eps['Endpoint'].tolist())
                    print(f'  {metric}: {ep_list}')

## 11. Summary

In [ ]:
print('='*80)
print('FINAL SUMMARY')
print('='*80)

# 1. Per-model macro metrics
print('\n1. MACRO-AVERAGED PERFORMANCE')
print('-'*40)
for model_name in predictions:
    eps = list(point_metrics.get(model_name, {}).keys())
    if not eps:
        continue
    print(f'\n{model_name} ({len(eps)} endpoints):')
    for metric in ['MAE', 'R2', 'Spearman']:
        vals = [point_metrics[model_name][ep][metric][0] for ep in eps]
        print(f'  {metric:10s}: {np.mean(vals):.4f}')

# 2. Pretraining effect
print('\n\n2. PRETRAINING EFFECT')
print('-'*40)
for title, df in improvement_dfs.items():
    n_better = (df['MAE_better'] == True).sum()
    n_total = len(df)
    avg_mae_pct = -df['MAE_pct'].mean()  # negate so positive = improvement
    avg_r2_delta = df['R2_delta'].mean()
    print(f'\n{title}:')
    print(f'  Endpoints improved (MAE): {n_better}/{n_total}')
    print(f'  Avg MAE improvement:      {avg_mae_pct:+.1f}%')
    print(f'  Avg R2 improvement:       {avg_r2_delta:+.4f}')

# 3. Best model per metric on LogD
print('\n\n3. BEST MODEL FOR LogD')
print('-'*40)
logd_models = [n for n in predictions if 'LogD' in point_metrics.get(n, {})]
for metric in ['MAE', 'R2', 'Spearman']:
    if metric in LOWER_IS_BETTER:
        best = min(logd_models, key=lambda n: point_metrics[n]['LogD'][metric][0])
    else:
        best = max(logd_models, key=lambda n: point_metrics[n]['LogD'][metric][0])
    mean, std = point_metrics[best]['LogD'][metric]
    print(f'  {metric:10s}: {best} = {mean:.4f} \u00b1 {std:.4f}')